# Predikcija kategorije proizvoda na osnovu naslova

**Autor:** Stefan Stojanović

## Cilj projekta

Cilj ovog projekta je razvoj modela mašinskog učenja koji na osnovu naziva proizvoda automatski predlaže odgovarajuću kategoriju.

Projekat obuhvata analizu i čišćenje podataka, inženjering karakteristika, treniranje i poređenje više modela, evaluaciju najboljeg rešenja i njegovo čuvanje za kasniju upotrebu.

Razvijeni model može doprineti bržem unosu proizvoda, smanjenju broja grešaka pri ručnoj kategorizaciji i jednostavnijem radu zaposlenih na platformi za online trgovinu.

## 1. Uvoz potrebnih biblioteka

U ovom delu uvozimo biblioteke potrebne za rad sa podacima, vizualizaciju rezultata i razvoj modela mašinskog učenja.

In [1]:
# Biblioteke za učitavanje i obradu podataka
import pandas as pd
import numpy as np

# Biblioteke za vizualizaciju
import matplotlib.pyplot as plt
import seaborn as sns

# Pomoćne biblioteke
import re
import time
from pathlib import Path

# Podešavanje prikaza tabela
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

# Podešavanje izgleda grafikona
sns.set_theme(style="whitegrid")

print("Sve biblioteke su uspešno učitane.")

Sve biblioteke su uspešno učitane.


## 2. Učitavanje i početni pregled podataka

Učitavamo skup podataka iz foldera `data` i proveravamo njegove dimenzije i prve redove. Putanja je podešena tako da sveska može da radi i kada se pokrene iz glavnog foldera projekta i kada se pokrene iz foldera `notebooks`.

In [2]:
# Moguće putanje do skupa podataka
possible_paths = [
    Path("../data/products.csv"),
    Path("data/products.csv")
]

# Pronalazimo prvu postojeću putanju
data_path = next((path for path in possible_paths if path.exists()), None)

# Zaustavljamo izvršavanje uz jasnu poruku ako fajl nije pronađen
if data_path is None:
    raise FileNotFoundError(
        "Fajl products.csv nije pronađen u folderu data."
    )

# Učitavanje podataka
df = pd.read_csv(data_path)

print(f"Dataset je uspešno učitan sa putanje: {data_path}")
print(f"Broj redova: {df.shape[0]}")
print(f"Broj kolona: {df.shape[1]}")

# Prikaz prvih pet redova
df.head()

Dataset je uspešno učitan sa putanje: ..\data\products.csv
Broj redova: 35311
Broj kolona: 8


,product ID,Product Title,Merchant ID,Category Label,_Product Code,Number_of_Views,Merchant Rating,Listing Date
0,1,apple iphone 8 plus 64gb silver,1,Mobile Phones,QA-2276-XC,860.0,2.5,5/10/2024
1,2,apple iphone 8 plus 64 gb spacegrau,2,Mobile Phones,KA-2501-QO,3772.0,4.8,12/31/2024
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim free smartphone in gold,3,Mobile Phones,FP-8086-IE,3092.0,3.9,11/10/2024
3,4,apple iphone 8 plus 64gb space grey,4,Mobile Phones,YI-0086-US,466.0,3.4,5/2/2022
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked sim free,5,Mobile Phones,NZ-3586-WP,4426.0,1.6,4/12/2023


### 2.1. Struktura skupa i tipovi podataka

Proveravamo tačne nazive kolona, tipove podataka i broj popunjenih vrednosti. Ovaj pregled pomaže da otkrijemo skrivene razmake u nazivima kolona i kolone koje sadrže nedostajuće podatke.

In [3]:
# Prikaz tačnih naziva kolona
print("Nazivi kolona:")
for column in df.columns:
    print(repr(column))

print("\nOsnovne informacije o skupu podataka:")
df.info()

Nazivi kolona:
'product ID'
'Product Title'
'Merchant ID'
' Category Label'
'_Product Code'
'Number_of_Views'
'Merchant Rating'
' Listing Date  '

Osnovne informacije o skupu podataka:
<class 'pandas.DataFrame'>
RangeIndex: 35311 entries, 0 to 35310
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   product ID       35311 non-null  int64  
 1   Product Title    35139 non-null  str    
 2   Merchant ID      35311 non-null  int64  
 3    Category Label  35267 non-null  str    
 4   _Product Code    35216 non-null  str    
 5   Number_of_Views  35297 non-null  float64
 6   Merchant Rating  35141 non-null  float64
 7    Listing Date    35252 non-null  str    
dtypes: float64(2), int64(2), str(4)
memory usage: 2.2 MB


### 2.2. Nedostajuće vrednosti i duplikati

Proveravamo broj i procenat nedostajućih vrednosti u svakoj koloni, kao i postojanje potpuno dupliranih redova. Ovi rezultati će odrediti naredne korake čišćenja podataka.

In [4]:
# Broj nedostajućih vrednosti po kolonama
missing_values = df.isna().sum()

# Procenat nedostajućih vrednosti po kolonama
missing_percentage = (missing_values / len(df) * 100).round(2)

# Tabela sa rezultatima
missing_summary = pd.DataFrame({
    "Broj nedostajućih": missing_values,
    "Procenat (%)": missing_percentage
})

print("Pregled nedostajućih vrednosti:")
display(missing_summary)

# Provera potpuno dupliranih redova
duplicate_rows = df.duplicated().sum()

print(f"\nBroj potpuno dupliranih redova: {duplicate_rows}")

Pregled nedostajućih vrednosti:


,Broj nedostajućih,Procenat (%)
product ID,0,0.00
Product Title,172,0.49
Merchant ID,0,0.00
Category Label,44,0.12
_Product Code,95,0.27
Number_of_Views,14,0.04
Merchant Rating,170,0.48
Listing Date,59,0.17



Broj potpuno dupliranih redova: 0


### 2.3. Analiza ciljne promenljive

Ciljna promenljiva je kategorija proizvoda. Proveravamo broj jedinstvenih kategorija, njihove nazive i broj proizvoda u svakoj kategoriji. Posebnu pažnju obraćamo na različito napisane oznake koje možda predstavljaju istu kategoriju.

In [5]:
# Pronalaženje kolone sa kategorijama bez oslanjanja na skrivene razmake
category_column = next(
    column for column in df.columns
    if column.strip() == "Category Label"
)

# Broj jedinstvenih nepraznih kategorija
number_of_categories = df[category_column].nunique()

print(f"Broj jedinstvenih nepraznih kategorija: {number_of_categories}")

print("\nTačni nazivi kategorija:")
for category in sorted(df[category_column].dropna().unique()):
    print(repr(category))

print("\nBroj proizvoda po kategorijama:")
display(df[category_column].value_counts(dropna=False).to_frame("Broj proizvoda"))

Broj jedinstvenih nepraznih kategorija: 13

Tačni nazivi kategorija:
'CPU'
'CPUs'
'Digital Cameras'
'Dishwashers'
'Freezers'
'Fridge Freezers'
'Fridges'
'Microwaves'
'Mobile Phone'
'Mobile Phones'
'TVs'
'Washing Machines'
'fridge'

Broj proizvoda po kategorijama:


,Broj proizvoda
Category Label,
Fridge Freezers,5495
Washing Machines,4036
Mobile Phones,4020
CPUs,3771
TVs,3564
Fridges,3457
Dishwashers,3418
Digital Cameras,2696
Microwaves,2338


## 3. Čišćenje i standardizacija podataka

Tokom početnog pregleda pronađeni su suvišni razmaci u nazivima kolona, prazni naslovi i kategorije, kao i različito napisane oznake za iste kategorije.

Nazive kolona standardizujemo radi jednostavnijeg i pouzdanijeg rada. Uklanjamo samo redove kojima nedostaje naslov proizvoda ili ciljna kategorija, jer su upravo te dve kolone neophodne za treniranje modela.

Nedostajuće vrednosti u ostalim kolonama ne zahtevaju uklanjanje redova, pošto model kategoriju predviđa na osnovu naslova proizvoda.

In [6]:
# Kreiranje kopije originalnog skupa
df_clean = df.copy()

# Standardizacija naziva kolona
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Broj redova pre čišćenja
initial_row_count = len(df_clean)

# Uklanjanje redova bez naslova ili kategorije
df_clean = df_clean.dropna(
    subset=["product_title", "category_label"]
).copy()

# Uklanjanje suvišnih razmaka iz tekstualnih vrednosti
df_clean["product_title"] = (
    df_clean["product_title"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df_clean["category_label"] = df_clean["category_label"].str.strip()

# Objedinjavanje različitih oznaka koje predstavljaju iste kategorije
category_mapping = {
    "CPU": "CPUs",
    "Mobile Phone": "Mobile Phones",
    "fridge": "Fridges"
}

df_clean["category_label"] = (
    df_clean["category_label"]
    .replace(category_mapping)
)

# Uklanjanje eventualnih naslova koji sadrže samo razmake
df_clean = df_clean[
    df_clean["product_title"].str.len() > 0
].copy()

# Resetovanje indeksa nakon uklanjanja redova
df_clean = df_clean.reset_index(drop=True)

removed_row_count = initial_row_count - len(df_clean)

print("Standardizovani nazivi kolona:")
print(df_clean.columns.tolist())

print(f"\nBroj redova pre čišćenja: {initial_row_count}")
print(f"Broj uklonjenih redova: {removed_row_count}")
print(f"Broj redova nakon čišćenja: {len(df_clean)}")
print(f"Broj kategorija nakon standardizacije: {df_clean['category_label'].nunique()}")

Standardizovani nazivi kolona:
['product_id', 'product_title', 'merchant_id', 'category_label', 'product_code', 'number_of_views', 'merchant_rating', 'listing_date']

Broj redova pre čišćenja: 35311
Broj uklonjenih redova: 215
Broj redova nakon čišćenja: 35096
Broj kategorija nakon standardizacije: 10


### 3.1. Analiza ponovljenih i konfliktnih naslova

Isti proizvod može biti unet više puta, zbog čega proveravamo ponovljene normalizovane naslove. Posebno proveravamo slučajeve u kojima je potpuno isti naslov povezan sa različitim kategorijama.

Takvi konflikti mogu zbuniti model, jer isti ulaz tada ima više različitih ciljnih vrednosti. Normalizovana verzija naslova koristi se samo za otkrivanje ponavljanja i konflikata.

In [7]:
# Kreiranje normalizovane verzije naslova
df_clean["normalized_title"] = (
    df_clean["product_title"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Broj dodatnih pojavljivanja već viđenih naslova
duplicate_title_count = df_clean["normalized_title"].duplicated().sum()

# Broj svih redova koji pripadaju ponovljenim naslovima
rows_with_duplicate_titles = (
    df_clean["normalized_title"]
    .duplicated(keep=False)
    .sum()
)

# Broj različitih kategorija za svaki normalizovani naslov
categories_per_title = (
    df_clean.groupby("normalized_title")["category_label"]
    .nunique()
)

# Naslovi povezani sa više različitih kategorija
conflicting_titles = categories_per_title[
    categories_per_title > 1
].index

conflicting_rows = df_clean[
    df_clean["normalized_title"].isin(conflicting_titles)
].sort_values("normalized_title")

print(f"Broj dodatnih pojavljivanja ponovljenih naslova: {duplicate_title_count}")
print(f"Broj svih redova sa ponovljenim naslovima: {rows_with_duplicate_titles}")
print(f"Broj naslova povezanih sa više kategorija: {len(conflicting_titles)}")
print(f"Broj redova obuhvaćenih konfliktima: {len(conflicting_rows)}")

print("\nPrimeri konfliktnih naslova:")
display(
    conflicting_rows[
        ["product_title", "category_label"]
    ].head(20)
)

Broj dodatnih pojavljivanja ponovljenih naslova: 4272
Broj svih redova sa ponovljenim naslovima: 7424
Broj naslova povezanih sa više kategorija: 2
Broj redova obuhvaćenih konfliktima: 4

Primeri konfliktnih naslova:


,product_title,category_label
31510,blomberg kgm9550 fridge freezer,Fridge Freezers
34981,blomberg kgm9550 fridge freezer,Fridges
23524,bosch kur15a50gb integrated undercounter fridge,Washing Machines
31538,bosch kur15a50gb integrated undercounter fridge,Fridges


### 3.2. Uklanjanje konfliktnih i ponovljenih naslova

Pronađena su dva normalizovana naslova povezana sa različitim kategorijama. Pošto na osnovu dostupnih podataka nije moguće pouzdano odrediti ispravnu oznaku, uklanjamo sve redove povezane sa tim naslovima.

Nakon toga svaki ponovljeni normalizovani naslov zadržavamo samo jednom. Identični naslovi ne pružaju novu tekstualnu informaciju modelu, a njihovo pojavljivanje i u trening i u test skupu moglo bi veštački povećati rezultat evaluacije.

In [8]:
# Broj redova pre uklanjanja konflikata i ponavljanja
rows_before_deduplication = len(df_clean)

# Uklanjanje svih redova sa konfliktnim naslovima
df_clean = df_clean[
    ~df_clean["normalized_title"].isin(conflicting_titles)
].copy()

rows_after_conflict_removal = len(df_clean)
removed_conflicting_rows = (
    rows_before_deduplication - rows_after_conflict_removal
)

# Zadržavanje samo jednog reda za svaki normalizovani naslov
df_clean = df_clean.drop_duplicates(
    subset="normalized_title",
    keep="first"
).copy()

removed_duplicate_rows = (
    rows_after_conflict_removal - len(df_clean)
)

# Resetovanje indeksa
df_clean = df_clean.reset_index(drop=True)

print(f"Uklonjeni konfliktni redovi: {removed_conflicting_rows}")
print(f"Uklonjeni ponovljeni redovi: {removed_duplicate_rows}")
print(f"Konačan broj redova: {len(df_clean)}")
print(
    "Preostali ponovljeni naslovi:",
    df_clean["normalized_title"].duplicated().sum()
)
print(
    "Nedostajući naslovi:",
    df_clean["product_title"].isna().sum()
)
print(
    "Nedostajuće kategorije:",
    df_clean["category_label"].isna().sum()
)
print(
    "Broj kategorija:",
    df_clean["category_label"].nunique()
)

Uklonjeni konfliktni redovi: 4
Uklonjeni ponovljeni redovi: 4270
Konačan broj redova: 30822
Preostali ponovljeni naslovi: 0
Nedostajući naslovi: 0
Nedostajuće kategorije: 0
Broj kategorija: 10
